# 07: Paired L9 experiment and checkpoint diagnostics

This notebook compares learning task 9 with and without the earlier U3 request. It then checks how much task 0's generated weights and BatchNorm statistics changed during L9. The two arms use separate saved checkpoints and run back to back in one session.

RUN_TRAINING and PAIR_ID in SETUP-02 control the work. Their current values open the completed pair for diagnostics without retraining. Set RUN_TRAINING to True and PAIR_ID to None to run a new pair. If Colab stops, copy the printed pair ID into PAIR_ID and rerun with RUN_TRAINING still True. Training needs a GPU. A saved pair can be diagnosed on CPU. Results are saved to Drive before the runtime is released.

## SETUP-01: Load repository code


In [ ]:
from urllib.request import urlretrieve
import runpy

urlretrieve(
    "https://raw.githubusercontent.com/sumitasthana/CARK/main/scripts/colab_setup.py",
    "/content/uncle_colab_setup.py",
)
_ = runpy.run_path("/content/uncle_colab_setup.py", run_name="__main__")


## SETUP-02: Choose a saved pair or start a new one

The two source checkpoints are the state after L3, L0 and the state after L3, L0, U3. This cell checks their histories and learning settings. Keep RUN_TRAINING False to diagnose a saved pair. Set it True and PAIR_ID None for a new pair. To resume an interrupted pair, keep RUN_TRAINING True and set PAIR_ID to the printed ID.


In [ ]:
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import torch
from google.colab import drive
from uncle.config import Config
from uncle.notebook import inspect_checkpoint

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace")
E08 = ROOT / "before_forgetting.pt"
POST_U3 = ROOT / "full_sequence_27step_candidate/checkpoint_seq1_resnet50_seed0_after_request02_forget3.pt"
DATA = Path("/content/data/tiny-imagenet-200")
PAIR_ROOT = ROOT / "paired_L9_probe/same_runtime_paired"
PAIR_ID = "20261003_233622_056045"  # Set to None for a new paired run.
RUN_TRAINING = False  # Set to True to run or resume training.
FOCUS_TASK = "0"
NEW_TASK = "9"

e08 = inspect_checkpoint(
    E08, expected_history=[("learn", "3"), ("learn", "0")],
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)
post = inspect_checkpoint(
    POST_U3, expected_history=[("learn", "3"), ("learn", "0"), ("forget", "3")],
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)
assert e08["forgotten"] == [] and post["forgotten"] == ["3"]
for key in ("dataset", "backbone", "seed", "chunks", "hidden", "code_dim",
            "classes_per_task", "epochs", "batch_size", "learning_rate", "beta"):
    assert e08["config"][key] == post["config"][key], key

full_config = Config(**post["config"])
assert full_config.requests[:4] == (
    ("learn", "3"), ("learn", "0"), ("forget", "3"), ("learn", NEW_TASK))
without_u3 = replace(
    full_config, requests=(("learn", "3"), ("learn", "0"), ("learn", NEW_TASK)))
after_u3 = replace(full_config, requests=full_config.requests[:4])
arms = {"without_u3": (E08, without_u3), "after_u3": (POST_U3, after_u3)}

run_new = RUN_TRAINING
if run_new and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime to run a new pair.")
pair_id = (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
           if PAIR_ID is None else PAIR_ID)
pair_dir = PAIR_ROOT / pair_id
if not run_new:
    assert pair_dir.is_dir(), pair_dir
print("Mode:", "run or resume paired training" if run_new else "diagnose saved pair")
print("Pair folder:", pair_dir)
print("Task 0 before U3:", e08["accuracies"][FOCUS_TASK])
print("Task 0 after U3:", post["accuracies"][FOCUS_TASK])


## RUN-01: Run or open the paired L9 experiment

With RUN_TRAINING True, each arm starts from its own source checkpoint and learns task 9 once. Each arm has a checkpoint and can resume after a disconnect. With RUN_TRAINING False, this cell reads the saved comparison and does not train.


In [ ]:
import hashlib
import json
import os
from uncle.experiments import run_experiment

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

comparison_path = pair_dir / "comparison.json"
if run_new:
    results = {}
    for name, (source, config) in arms.items():
        folder = pair_dir / name
        print("Learning task", NEW_TASK, "in arm", name, flush=True)
        results[name] = run_experiment(
            sequence=1, config=config, starting_checkpoint=source,
            root=DATA, download=True, output=folder,
            checkpoint=True, resume=True, verbose=True, progress=True,
        )

    comparison = {
        "schema_version": 1, "pair_id": pair_id,
        "focus_task": FOCUS_TASK, "new_task": NEW_TASK,
        "arms": {}, "checks": {},
    }
    runtimes = {}
    for name, (source, config) in arms.items():
        folder = pair_dir / name
        stem = f"seq1_{config.backbone}_seed{config.seed}"
        result = results[name]
        assert len(result["history"]) == len(config.requests)
        row = result["history"][-1]
        assert (row["action"], row["task"]) == ("learn", NEW_TASK)
        origin = json.loads(
            (folder / f"origin_{stem}.json").read_text(encoding="utf-8"))
        env = result["environment"]
        runtimes[name] = {
            key: env.get(key)
            for key in ("torch", "cuda", "gpu", "device", "platform", "git_commit")
        }
        comparison["arms"][name] = {
            "focus_before": row["before"][FOCUS_TASK],
            "focus_after": row["after"][FOCUS_TASK],
            "focus_loss_points": row["before"][FOCUS_TASK] - row["after"][FOCUS_TASK],
            "new_task_after": row["after"][NEW_TASK],
            "source_checkpoint": str(source),
            "source_sha256": origin["starting_checkpoint_sha256"],
            "source_sha256_matches": (
                origin["starting_checkpoint_sha256"] == sha256_file(source)),
            "runtime": runtimes[name],
        }
    comparison["checks"] = {
        "both_source_hashes_match": all(
            arm["source_sha256_matches"] for arm in comparison["arms"].values()),
        "same_recorded_runtime": runtimes["without_u3"] == runtimes["after_u3"],
    }
    comparison["extra_focus_loss_after_u3_points"] = (
        comparison["arms"]["after_u3"]["focus_loss_points"]
        - comparison["arms"]["without_u3"]["focus_loss_points"])
    temporary = comparison_path.with_suffix(".json.writing")
    temporary.write_text(json.dumps(comparison, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary, comparison_path)
else:
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    assert set(comparison["arms"]) == set(arms)
print("Paired comparison:", comparison_path)
print(json.dumps(comparison, indent=2))


## DIAG-01: Compare task 0 before and after L9

The hypernetwork produces the weights used by task 0. This cell reconstructs those weights from each saved checkpoint and measures how far they moved during L9. It also checks task 0's BatchNorm running statistics, task code and chunk codes. It does not train, estimate gradients or compute Fisher.


In [ ]:
import gc
import math
from uncle import checkpoint as checkpointing
from uncle.hypernet import HyperNetwork, build_target
from uncle.research_diagnostic import capture_components, compare_components

diagnostic_device = "cuda" if run_new else "cpu"

def snapshot(path, model=None):
    saved = checkpointing.load(path)
    if saved is None:
        raise FileNotFoundError(path)
    config = Config(**{**saved["config"], "device": diagnostic_device})
    if model is None:
        model = HyperNetwork(build_target(config), config)
    for task in saved["tasks_with_codes"]:
        if task not in model.task_codes:
            model.add_task(task)
    model.load_state_dict(saved["hypernet"])
    model.eval()
    with torch.no_grad():
        weights = model.weights_for(FOCUS_TASK)
        roles = {group: {} for group in model.group_sizes}
        for name, tensor in weights.items():
            roles[model.layout[name][0]][name] = tensor
        roles["task_normalization_buffers"] = saved["task_buffers"][FOCUS_TASK]
        components = capture_components(roles)
    meta = {
        "accuracy": saved["previous"][FOCUS_TASK],
        "task_code": saved["hypernet"][f"task_codes.{FOCUS_TASK}"].clone(),
        "chunk_codes": saved["hypernet"]["chunk_codes"].clone(),
    }
    del saved, weights, roles
    gc.collect()
    return components, meta, model

diagnostic = {
    "schema_version": 1, "pair_id": pair_id,
    "focus_task": FOCUS_TASK, "new_task": NEW_TASK,
    "arms": {},
}
for name, (start_path, config) in arms.items():
    stem = f"seq1_{config.backbone}_seed{config.seed}"
    end_path = pair_dir / name / f"checkpoint_{stem}.pt"
    before, start, model = snapshot(start_path)
    after, end, model = snapshot(end_path, model)
    changes = compare_components(before, after)
    weight_changes = {}
    for group in ("batchnorm", "residual", "weights"):
        baseline_l2 = math.sqrt(sum(
            tensor.float().square().sum().item()
            for tensor in before[group].values()))
        weight_changes[group] = {
            **changes[group],
            "relative_l2_pct": (
                100 * changes[group]["l2_change"] / baseline_l2
                if baseline_l2 else None),
        }
    diagnostic["arms"][name] = {
        "focus_accuracy_before": start["accuracy"],
        "focus_accuracy_after": end["accuracy"],
        "generated_weights": weight_changes,
        "task_normalization_buffers": changes["task_normalization_buffers"],
        "task_code_unchanged": torch.equal(start["task_code"], end["task_code"]),
        "chunk_codes_unchanged": torch.equal(
            start["chunk_codes"], end["chunk_codes"]),
    }
    del before, after, start, end, model
    gc.collect()
    if diagnostic_device == "cuda":
        torch.cuda.empty_cache()

diagnostic_path = pair_dir / "component_changes.json"
temporary = diagnostic_path.with_suffix(".json.writing")
temporary.write_text(json.dumps(diagnostic, indent=2) + "\n", encoding="utf-8")
os.replace(temporary, diagnostic_path)
for name, arm in diagnostic["arms"].items():
    print(name, "task", FOCUS_TASK, "accuracy:",
          arm["focus_accuracy_before"], "to", arm["focus_accuracy_after"])
    for group, change in arm["generated_weights"].items():
        relative = change["relative_l2_pct"]
        shown = "n/a" if relative is None else f"{relative:.4f}%"
        print(" ", group, "relative weight change:", shown)
    buffers = arm["task_normalization_buffers"]
    print("  BatchNorm buffers changed:",
          buffers["changed_tensors"], "of", buffers["tensors"])
    print("  task code unchanged:", arm["task_code_unchanged"])
    print("  chunk codes unchanged:", arm["chunk_codes_unchanged"])
print("Saved:", diagnostic_path)


## FINISH-01: Verify files and release the runtime

This checks both arm folders and reports before flushing Drive. A new GPU run releases the GPU after the flush. If a required file is missing, the cell stops so it can be inspected.


In [ ]:
from google.colab import runtime

assert json.loads(comparison_path.read_text(encoding="utf-8")) == comparison
assert json.loads(diagnostic_path.read_text(encoding="utf-8")) == diagnostic
for name, (_, config) in arms.items():
    stem = f"seq1_{config.backbone}_seed{config.seed}"
    folder = pair_dir / name
    for kind in ("checkpoint", "origin", "environment",
                 "history", "costs", "summary"):
        suffix = ".pt" if kind == "checkpoint" else ".json"
        path = folder / f"{kind}_{stem}{suffix}"
        assert path.is_file() and path.stat().st_size > 0, path
print("Saved pair and component diagnostic:", pair_dir, flush=True)
drive.flush_and_unmount(timeout_ms=600000)
if torch.cuda.is_available():
    print("Drive flushed. Releasing GPU.", flush=True)
    runtime.unassign()
else:
    print("Drive flushed. CPU run complete.", flush=True)
